In [9]:
import os
from pathlib import Path
import random
from PIL import Image
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, infer_device
import torch
from datetime import datetime
import time
import json
import hashlib
import numpy as np

from abc import ABC, abstractmethod



## Helper functions

In [2]:
# the print takes a long time so inspect to use a native package or another solution to do this faster
def retrieve_image_batches(images_folder, batch_size, sample_size=None, random_state=None):
    """
    Yield batches of images directly from a given folder.

    Args:
        images_folder (str | Path): Path to folder containing images (searched recursively).
        batch_size (int): Number of images per batch.
        sample_size (int, optional): Limit total number of images to sample.
        random_state (int, optional): Seed for reproducible shuffling if needed.
    """
    exts = (".jpg", ".jpeg", ".png", ".bmp", ".gif", ".webp")
    paths = [p for p in Path(images_folder).glob("*") if p.suffix.lower() in exts]
    print(f"Found {len(paths)} image files")

    if random_state is not None:
        random.seed(random_state)
        random.shuffle(paths)
    if sample_size:
        paths = paths[:sample_size]

    for i in range(0, len(paths), batch_size):
        batch = paths[i:i + batch_size]
        yield [Image.open(p).convert("RGB").copy() for p in batch], [p.name for p in batch]

In [3]:
test_root = "../Data/tomatoes/images/val"   # adjust to your dataset path
for batch_images, names in retrieve_image_batches(test_root, batch_size=2, sample_size=4):
    print("Batch filenames:", names)
    print("Batch size:", len(batch_images))
    break  # only first batch for quick test

Found 1325 image files
Batch filenames: ['2024_01_17__12_08_00_939000000___04Z49__04T2Y_bunch_0.png', '2025_02_26__14_23_25_756000000___camera_0__camera_1_bunch_1.png']
Batch size: 2


In [5]:
def build_output_path(experiment, images_folder, model_name):
    parent = os.path.basename(os.path.dirname(os.path.dirname(images_folder)))
    base = os.path.splitext(os.path.basename(images_folder))[0]

    out_dir = os.path.join("..", "Results", experiment)
    os.makedirs(out_dir, exist_ok=True)

    out_file = os.path.join(out_dir, f"{parent}_{base}_{model_name}_predictions.json")
    return out_file


In [6]:
def process_prediction_boxes(
    boxes, scores, labels, img_id, img_h, img_w, categories_dict, annotations
):
    """
    Apply clamping, ordering, xyxy→xywh conversion, label lookup,
    and append valid COCO annotations to `annotations`.

    Parameters
    ----------
    boxes : list[list[float]]
        List of [x1, y1, x2, y2] boxes.
    scores : list[float]
    labels : list[str]
    img_id : int
    img_h, img_w : int
        Image height and width.
    categories_dict : dict[str, int]
        Mapping from label string to category_id.
    annotations : list[dict]
        The list that will be appended to.

    Returns
    -------
    int
        Number of new valid boxes added.
    """
    added = 0

    for box, score, label in zip(boxes, scores, labels):
        x1, y1, x2, y2 = map(float, box)

        # Clamp to image bounds
        x1 = max(0.0, min(x1, img_w))
        y1 = max(0.0, min(y1, img_h))
        x2 = max(0.0, min(x2, img_w))
        y2 = max(0.0, min(y2, img_h))

        # Enforce ordering
        if x2 < x1:
            x1, x2 = x2, x1
        if y2 < y1:
            y1, y2 = y2, y1

        w = max(0.0, x2 - x1)
        h = max(0.0, y2 - y1)
        if w == 0.0 or h == 0.0:
            continue

        cid = categories_dict.get(label)
        if cid is None:
            continue

        ann_id = len(annotations) + 1
        annotations.append({
            "id": ann_id,
            "image_id": img_id,
            "category_id": cid,
            "bbox": [x1, y1, w, h],
            "score": float(score),
        })

        added += 1

    return added


In [7]:
def write_coco_output(
 images_folder,
 model_name,
 categories_list,
 images,
 annotations,
 num_images,
 num_boxes,
 total_time,
 gpu_hourly_price=2.5,
 gpu_tdp_watts=250.0,
 gpu_utilization_factor=0.9,
 power_utilization_factor=0.7,
 electricity_price_per_kwh=0.30,
):
 # ---- timing statistics ----
 avg_time_image = total_time / num_images if num_images else 0.0
 avg_time_bbox = total_time / num_boxes if num_boxes else 0.0

 # ---- cost / energy estimates ----
 gpu_hours = (total_time / 3600.0) * gpu_utilization_factor
 infra_cost_eur = gpu_hours * gpu_hourly_price
 energy_kwh = (gpu_tdp_watts / 1000.0) * (total_time / 3600.0) * power_utilization_factor
 energy_cost_eur = energy_kwh * electricity_price_per_kwh
 total_cost_eur = infra_cost_eur + energy_cost_eur

#  cost_per_image_eur = total_cost_eur / num_images if num_images else 0.0
 cost_per_bbox_eur = total_cost_eur / num_boxes if num_boxes else 0.0

 info = {
     "description": f"Predictions on {images_folder} with {model_name}",
     "date_created": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
     "num_images": num_images,
     "num_predicted_bbox": num_boxes,
     "avg_inference_time_s_image": avg_time_image,
     "avg_inference_time_s_bbox": avg_time_bbox,
     "total_inference_time_s": total_time,
     "gpu_hours_estimate": gpu_hours,
     "total_cost_eur": total_cost_eur,
     "cost_per_bbox_eur": cost_per_bbox_eur,
 }

 categories = [{"id": i + 1, "name": name} for i, name in enumerate(categories_list)]

 coco_output = {
     "info": info,
     "images": images,
     "annotations": annotations,
     "categories": categories,
 }

 out_file = build_output_path("Experiment_1", images_folder, model_name)
 with open(out_file, "w", encoding="utf-8") as f:
     json.dump(coco_output, f, indent=2)

 return out_file

In [191]:
def select_model(model_name):
    if model_name == "gd_t":
        model_id = "IDEA-Research/grounding-dino-tiny"

    if model_name == "gd_b":
        model_id = "IDEA-Research/grounding-dino-base"

    if model_name == "owlvit_b_16":
        model_id = "google/owlvit-base-patch16"

    if model_name == "owlvit_b_32":
        model_id = "google/owlvit-base-patch32"

    if model_name == "owlv2_16_ensemble": # crashes RAM
        model_id = "owlv2-base-patch16-ensemble"

    if model_name == "mmgd_t":
        model_id = "openmmlab-community/mm_grounding_dino_tiny_o365v1_goldg_v3det"

    if model_name == "mmgd_b_all": # too big for T4 GPU
        model_id = "rziga/mm_grounding_dino_base_all"

    if model_name == "mmgd_l_all": # too big for T4 GPU
        model_id = "rziga/mm_grounding_dino_large_all"

    device = infer_device()
    processor = AutoProcessor.from_pretrained(model_id)#, token=os.environ["HF_TOKEN"])
    model = AutoModelForZeroShotObjectDetection.from_pretrained(model_id)#, token=os.environ["HF_TOKEN"]).to(device)

    return processor, model

## Define backends

In [203]:
class ZeroShotBackend(ABC):
    @abstractmethod
    def build_text_inputs_cache(self, processor, model, categories_list):
        """
        Return a dict of text tensors on the correct device, ready to be
        repeated per batch. Can be None if the model doesn't use text.
        """
        ...

    @abstractmethod
    def build_batch_inputs(self, model, batch_size_actual, text_inputs_cache, inputs):
        """
        Return a dict of model inputs for this batch, moved to the correct device.
        """
        ...

    @abstractmethod
    def postprocess(
        self,
        processor,
        outputs,
        inputs,
        batch_images,
        text_labels,
        threshold,
        text_threshold,
    ):
        """
        Return a list of per-image results in a common format:
        [
            {
                "boxes": Tensor[N,4],
                "scores": Tensor[N],
                "labels": list[str] or Tensor[N]
            },
            ...
        ]
        """
        ...


In [201]:
class GroundedDetBackend(ZeroShotBackend):
    def build_text_inputs_cache(self, processor, model, categories_list):
        # GroundingDINO expects: "cat. dog. person."
        text_labels = ". ".join(categories_list) + "."

        text_inputs = processor(
            text=text_labels,
            return_tensors="pt",
            padding=True,
        )

        text_inputs = {
            k: v.to(model.device, non_blocking=True)
            for k, v in text_inputs.items()
        }
        return text_inputs,text_labels # pass text_labels for consistency with other methods (grounding dino takes the tokens in the postprocessing step and does not need text_labels)

    def build_batch_inputs(self, model, batch_size_actual, text_inputs_cache, inputs):

        # add cached text encoding (2D -> [B, L])
        if text_inputs_cache is not None:
            for k, v in text_inputs_cache.items():
                inputs[k] = v.repeat(batch_size_actual, 1)

        # move everything to device
        inputs = {
            k: v.to(model.device, non_blocking=True)
            for k, v in inputs.items()
        }
        return inputs

    def postprocess(
        self,
        processor,
        outputs,
        inputs,
        batch_images,
        text_labels,  # unused but kept for consistency
        threshold=None,
        text_threshold=None,
    ):
        target_sizes = [(im.height, im.width) for im in batch_images]

        kwargs = {
            "target_sizes": target_sizes,
        }
        if threshold is not None:
            kwargs["threshold"] = threshold
        if text_threshold is not None:
            kwargs["text_threshold"] = text_threshold

        return processor.post_process_grounded_object_detection(
            outputs,
            inputs["input_ids"],
            **kwargs,
        )


In [208]:
class OwlViTBackend(ZeroShotBackend):
    def build_text_inputs_cache(self, processor, model, categories_list):
        # Encode each category as a separate sentence
        text_labels = [f"a photo of a {c}" for c in categories_list]
        text_inputs = processor(
            text=text_labels,
            return_tensors="pt",
            padding=True,
        )
        text_inputs = {k: v.to(model.device, non_blocking=True)
                       for k, v in text_inputs.items()}
        return text_inputs,text_labels

    def build_batch_inputs(self, model, batch_size_actual, text_inputs_cache, inputs):
        # pixel_values are already in `inputs["pixel_values"]` with shape [B, 3, H, W]

        if text_inputs_cache is not None:
            for k, v in text_inputs_cache.items():
                # v: [Q, L]
                if v.dim() != 2:
                    raise ValueError(f"Expected 2D text tensor for {k}, got {v.shape}")

                # Repeat text queries for each image:
                # [Q, L] -> [B*Q, L]
                v = v.repeat(batch_size_actual, 1)
                inputs[k] = v.to(model.device, non_blocking=True)

        # Make sure everything else is on device (including pixel_values)
        inputs = {
            k: v.to(model.device, non_blocking=True)
            for k, v in inputs.items()
        }
        print(inputs)
        return inputs

    def postprocess(
        self,
        processor,
        outputs,
        inputs,  # unused but kept for interface consistency
        batch_images,
        text_labels,
        threshold=None,
        text_threshold=None,  # unused but kept for interface consistency
    ):
        target_sizes = [(im.height, im.width) for im in batch_images]

        kwargs = {
            "target_sizes": target_sizes,
        }
        if threshold is not None:
            kwargs["threshold"] = threshold

        return processor.post_process_grounded_object_detection(
            outputs,
            text_labels=[text_labels] * len(batch_images),
            **kwargs,
        )



In [ ]:
BACKENDS = {
    "grounded_model": GroundedDetBackend(),
    "owl_model": OwlViTBackend(),
    # ...
}

# Map all model names from select_model() to a backend identifier.
MODEL_BACKEND_MAP = {
    "gd_t": "grounded_model",
    "gd_b": "grounded_model",
    "owlvit_b_16": "owl_model",
    "owlvit_b_32": "owl_model",
    "owlv2_16_ensemble": "owl_model",
    "mmgd_t": "grounded_model",
    "mmgd_b_all": "grounded_model",
    "mmgd_l_all": "grounded_model",
}

def get_backend(model_name: str) -> ZeroShotBackend:
    try:
        backend_key = MODEL_BACKEND_MAP[model_name]
    except KeyError:
        raise ValueError(f"No backend registered for model_name={model_name!r}")

    try:
        return BACKENDS[backend_key]
    except KeyError:
        raise ValueError(f"No backend instance found for backend_key={backend_key!r}")


## prediction function

In [206]:
def make_zero_shot_predictions(
    images_folder,
    categories_list,
    model_name,
    batch_size,
    sample_size,
    threshold = None,
    text_threshold = None,
    random_state=None,

):
    """
    Run zero-shot object detection and export results in COCO format.

    This function:
      1. Loads a model and processor via `select_model(model_name)`.
      2. Iterates over images in `images_folder` in batches.
      3. Runs zero-shot detection.
      4. Collects predictions into COCO-style structures.
      5. Calls `write_coco_output` to write a JSON file for Experiment_1.

    Parameters
    ----------
    images_folder : str
        Folder containing images.
    categories_list : list[str]
        Class names (e.g. ["cat", "dog", "person"]).
    model_name : str
        Model identifier for select_model().
    batch_size : int
        Number of images per batch.
    sample_size : int
        Total number of images to sample from the folder.
    random_state : int or None, optional
        Random seed used in `retrieve_image_batches`.
    threshold : float, optional
        Detection score threshold.
    text_threshold : float, optional
        Text matching threshold for the grounded detection head.
    """

    # -------------------------------------------------------------------------
    # Setup
    # -------------------------------------------------------------------------
    processor, model = select_model(model_name)
    model.eval() # set the model in inference mode

    use_cuda = (model.device.type == "cuda")

    backend = get_backend(model_name)

    # Map category name -> category id (1-based)
    categories_dict = {name: i + 1 for i, name in enumerate(categories_list)}

    images = []
    annotations = []

    def image_id_from_name(name: str) -> int:
        """Stable integer id derived from the image filename."""
        return int(hashlib.md5(name.encode()).hexdigest()[:8], 16)

    # Labels used for the text encoder
    # label_list = categories_list
    text_inputs_cache  = None  # filled the first time we see a batch

    # Simple timing variables
    total_time = 0.0
    num_images = 0
    num_boxes = 0

    warmup_steps = batch_size  # number of warmup steps
    first_batch = True

    num_forwards = 0  # add at top of make_zero_shot_predictions

    # ------------------------------------------------------------------------- 
    # Main loop: iterate over image batches 
    # ------------------------------------------------------------------------- 
    with torch.inference_mode():
        for batch_images, names in retrieve_image_batches(
            images_folder=images_folder,
            batch_size=batch_size,
            sample_size=sample_size,
            random_state=random_state,
        ):
            # print(f"[DEBUG] Got batch with {len(batch_images)} images, names={names}")

            # 1) model-specific text encoding (once)
            if text_inputs_cache is None:
                text_inputs_cache, text_labels = backend.build_text_inputs_cache(
                    processor=processor,
                    model=model,
                    categories_list=categories_list,
                )

            # 2) common image preprocessing
            inputs = processor(
                images=batch_images,
                return_tensors="pt",
                padding=True,
            )
            # print("[DEBUG] pixel_values shape:", inputs["pixel_values"].shape)

            batch_size_actual = len(batch_images)

            # 3) model-specific addition of cached text encoding
            inputs = backend.build_batch_inputs(
                model=model,
                batch_size_actual=batch_size_actual,
                text_inputs_cache=text_inputs_cache,
                inputs=inputs,
            )

            # 4) common warmup & inference
            if first_batch:
                print("[DEBUG] Doing warmup")
                for _ in range(warmup_steps):
                    _ = model(**inputs)
                    if use_cuda:
                        torch.cuda.synchronize()
                first_batch = False

            num_forwards += 1
            print(f"[DEBUG] Running forward pass #{num_forwards}")
            # forward
            start = time.perf_counter()
            outputs = model(**inputs)
            if use_cuda:
                torch.cuda.synchronize()
            end = time.perf_counter()
            print(f"[DEBUG] Forward pass took {end - start:.4f} seconds")

            total_time += (end - start)
            num_images += len(batch_images)

            # 5) model-specific postprocess
            results = backend.postprocess(
                processor=processor,
                outputs=outputs,
                inputs=inputs,
                batch_images=batch_images,
                text_labels=text_labels,
                threshold=threshold,
                text_threshold=text_threshold,
            )

            print(f"[DEBUG] num results for batch: {len(results)}")
            for name, res in zip(names, results):
                num_boxes_res = len(res.get("boxes", []))
                print(f"[DEBUG] Image {name}: {num_boxes_res} boxes")
                if num_boxes_res > 0:
                    print("[DEBUG] scores[:5]:", res["scores"][:5])

            for name, res in zip(names, results):
                boxes = res.get("boxes", [])
                scores = res.get("scores", [])
                if len(boxes) == 0:
                    print(f"[DEBUG] {name}: 0 boxes, maybe no confident detections")
                else:
                    print(f"[DEBUG] {name}: {len(boxes)} boxes, max score={scores.max():.3f}")

            # 6) common COCO conversion
            for name, res, im in zip(names, results, batch_images):
                img_id = image_id_from_name(name)
                H, W = im.height, im.width

                images.append({
                    "id": img_id,
                    "file_name": f"images/val/{name}",
                })

                boxes = res["boxes"].tolist()
                scores = res["scores"].tolist()
                labels = res.get("text_labels", [])

                num_boxes += process_prediction_boxes(
                    boxes=boxes,
                    scores=scores,
                    labels=labels,
                    img_id=img_id,
                    img_h=H,
                    img_w=W,
                    categories_dict=categories_dict,
                    annotations=annotations,
                )

    print(f"[DEBUG] num_images={num_images}, num_forwards={num_forwards}, total_time={total_time:.4f}s")
    # -------------------------------------------------------------------------
    # Write COCO JSON and log path
    # -------------------------------------------------------------------------
    out_file = write_coco_output(
        images_folder=images_folder,
        model_name=model_name,
        categories_list=categories_list,
        images=images,
        annotations=annotations,
        num_images=num_images,
        num_boxes=num_boxes,
        total_time=total_time,
    )

    print(f"Wrote COCO-format JSON to {out_file}")

In [ ]:
make_zero_shot_predictions(
    images_folder="../Data/tomatoes/images/val",
    categories_list=["tomato", "leaf", "stem"],
    model_name="gd_t",
    batch_size=2,
    sample_size=4,
    # threshold=0.25,
    # text_threshold=0.25,
)

Found 1325 image files
{'pixel_values': tensor([[[[ 1.1274,  1.0982,  1.0544,  ..., -0.5806, -0.5806, -0.5806],
          [ 1.1128,  1.0836,  1.0398,  ..., -0.5660, -0.5660, -0.5660],
          [ 1.0982,  1.0690,  1.0252,  ..., -0.5368, -0.5368, -0.5368],
          ...,
          [ 1.8865,  1.8865,  1.8865,  ...,  1.3318,  1.3172,  1.3026],
          [ 1.9011,  1.9011,  1.9011,  ...,  1.3610,  1.3464,  1.3318],
          [ 1.9157,  1.9157,  1.9157,  ...,  1.3902,  1.3756,  1.3610]],

         [[ 0.4540,  0.4390,  0.3940,  ..., -0.9267, -0.9267, -0.9267],
          [ 0.4390,  0.4240,  0.3790,  ..., -0.9117, -0.9117, -0.9117],
          [ 0.4090,  0.4090,  0.3640,  ..., -0.8816, -0.8816, -0.8816],
          ...,
          [ 0.9193,  0.9193,  0.9193,  ...,  0.1989,  0.1839,  0.1839],
          [ 0.9343,  0.9343,  0.9343,  ...,  0.1839,  0.1689,  0.1689],
          [ 0.9493,  0.9493,  0.9493,  ...,  0.1839,  0.1689,  0.1689]],

         [[ 0.3826,  0.3684,  0.3115,  ..., -0.8545, -0.8545, 

In [207]:
make_zero_shot_predictions(
    images_folder="../Data/tomatoes/images/val",
    categories_list=["tomato"],
    model_name="owlvit_b_32",
    batch_size=4,
    sample_size=4,
    # threshold=0.01,
    # text_threshold=0.3,
)

Found 1325 image files
{'pixel_values': tensor([[[[ 1.1274,  1.0982,  1.0544,  ..., -0.5806, -0.5806, -0.5806],
          [ 1.1128,  1.0836,  1.0398,  ..., -0.5660, -0.5660, -0.5660],
          [ 1.0982,  1.0690,  1.0252,  ..., -0.5368, -0.5368, -0.5368],
          ...,
          [ 1.8865,  1.8865,  1.8865,  ...,  1.3318,  1.3172,  1.3026],
          [ 1.9011,  1.9011,  1.9011,  ...,  1.3610,  1.3464,  1.3318],
          [ 1.9157,  1.9157,  1.9157,  ...,  1.3902,  1.3756,  1.3610]],

         [[ 0.4540,  0.4390,  0.3940,  ..., -0.9267, -0.9267, -0.9267],
          [ 0.4390,  0.4240,  0.3790,  ..., -0.9117, -0.9117, -0.9117],
          [ 0.4090,  0.4090,  0.3640,  ..., -0.8816, -0.8816, -0.8816],
          ...,
          [ 0.9193,  0.9193,  0.9193,  ...,  0.1989,  0.1839,  0.1839],
          [ 0.9343,  0.9343,  0.9343,  ...,  0.1839,  0.1689,  0.1689],
          [ 0.9493,  0.9493,  0.9493,  ...,  0.1839,  0.1689,  0.1689]],

         [[ 0.3826,  0.3684,  0.3115,  ..., -0.8545, -0.8545, 